# 1. GPT-2 Quantization Benchmark
This notebook measures a common FP32 baseline and optional bitsandbytes INT8/INT4 GPU variants. It does not claim that quantization is faster on every GPU.

In [ ]:
# Colab / local setup
%pip install -q -U transformers accelerate datasets peft evaluate bitsandbytes

import os, time, math, random, json, gc
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
# Quantization: GPT-2 FP32 baseline vs bitsandbytes INT8/INT4
# NOTE: bitsandbytes 8/4-bit paths are GPU-oriented. Exact latency depends on hardware.

MODEL_NAME="gpt2"
WORKDIR=Path("./lab2_quant_demo"); WORKDIR.mkdir(exist_ok=True)
from datasets import load_dataset
from transformers import BitsAndBytesConfig

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"

fp32=AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cpu").eval()

def model_size_mb(model):
    return sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)

def checkpoint_size_mb(model, path):
    path=Path(path); path.mkdir(exist_ok=True,parents=True); model.save_pretrained(path); tokenizer.save_pretrained(path)
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())/(1024**2)

# Perplexity on an unpadded contiguous token stream. This avoids counting padding as real text.
def ppl(model, text, max_length=512, stride=256):
    model.eval(); dev=next(model.parameters()).device
    enc=tokenizer(text, return_tensors="pt", add_special_tokens=False)
    ids=enc.input_ids[0]
    max_ctx=getattr(model.config,"n_positions",getattr(model.config,"max_position_embeddings",1024))
    max_length=min(max_length,max_ctx)
    nll=[]; total=0
    for begin in range(0, len(ids), stride):
        end=min(begin+max_length,len(ids)); begin_ctx=max(0,end-max_length)
        x=ids[begin_ctx:end].unsqueeze(0).to(dev)
        target=x.clone(); trg_len=end-begin
        target[:,:-trg_len]=-100
        with torch.no_grad(): out=model(x, labels=target)
        n=trg_len
        nll.append(out.loss.detach().float()*n); total+=n
        if end==len(ids): break
    return float(torch.exp(torch.stack(nll).sum()/total))

def latency(model,prompt,max_new_tokens=50,runs=3):
    dev=next(model.parameters()).device; x=tokenizer(prompt,return_tensors="pt").to(dev)
    for _ in range(1):
        with torch.no_grad(): model.generate(**x,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    ts=[]
    for _ in range(runs):
        if dev.type=="cuda": torch.cuda.synchronize()
        t=time.perf_counter()
        with torch.no_grad(): model.generate(**x,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        if dev.type=="cuda": torch.cuda.synchronize()
        ts.append(time.perf_counter()-t)
    return float(np.median(ts))

EVAL_TEXT="Artificial intelligence is transforming the world. "*200
prompt="Artificial intelligence is transforming the world of"
results=[]
results.append({"variant":"FP32","param_memory_mb":model_size_mb(fp32),"ppl":ppl(fp32,EVAL_TEXT),"latency_s":latency(fp32,prompt)})
print(results[-1])

if torch.cuda.is_available():
    for label,cfg in [("INT8",BitsAndBytesConfig(load_in_8bit=True)),("INT4-NF4",BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16))]:
        try:
            q=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=cfg,device_map="auto").eval()
            r={"variant":label,"param_memory_mb":model_size_mb(q),"ppl":ppl(q,EVAL_TEXT),"latency_s":latency(q,prompt)}
            results.append(r); print(r)
            del q; gc.collect(); torch.cuda.empty_cache()
        except Exception as e: print(label,"SKIPPED:",repr(e))
else:
    print("INT8/INT4 skipped: CUDA is required for this bitsandbytes experiment.")
